# quick check: pygam with sklearn tools

Just trying to see why pygam breaks with sklearn >=1.7

Found this issue:
https://github.com/dswah/pyGAM/issues/422

Apparently something related to sklearn_tags.

Not 100% sure yet so just testing a few things here.

In [ ]:
import sklearn
import numpy as np
from pygam import GAM

print("sklearn version:", sklearn.__version__)

First test: just create the estimator

In [ ]:
model = GAM()
model

In [ ]:
Now trying with RandomizedSearchCV

This is where the issue seems to appear according to the github thread.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import r2_score, make_scorer

In [ ]:
np.random.seed(0)

X = np.random.randn(120,3)
y = 2*X[:,0] + np.random.randn(120)

X[:5], y[:5]

In [ ]:
scorer = make_scorer(r2_score)

search = RandomizedSearchCV(
    model,
    cv=KFold(3),
    param_distributions={"max_iter":[50,100]},
    n_iter=2,
    scoring=scorer
)

Running fit.

Expecting something to break here if sklearn >=1.7

In [ ]:
search.fit(X,y)

ok so sklearn is calling something called __sklearn_tags__()

Looking quickly inside sklearn source it seems newer versions rely on it for estimator introspection.

pygam probably doesn't implement it.

In [ ]:
hasattr(model, "__sklearn_tags__")

Trying quick hack: subclass GAM and add minimal tag method

Just testing if sklearn accepts it.

In [ ]:
class PatchedGAM(GAM):

    def __sklearn_tags__(self):
        return {
            "requires_y": True,
            "allow_nan": False,
            "non_deterministic": False,
        }

patched_model = PatchedGAM()
patched_model

In [ ]:
search2 = RandomizedSearchCV(
    patched_model,
    cv=KFold(3),
    param_distributions={"max_iter":[50,100]},
    n_iter=2,
    scoring=scorer
)

In [ ]:
search2.fit(X,y)

This seems to run without the previous error.

So likely the missing sklearn_tags is the main compatibility issue.

Need to check later if additional estimator checks fail though.